This code creates a chatbot using Langchain framework. we wish tou use openAI LLM model in it.

In [ ]:
#to manage API key as a local enviornment variable we need OS library, and to load the env variables from .env file we need to install python-dotenv package
%pip install python-dotenv

import os
#load openai key from .env file, first import the library to load env variables
from dotenv import load_dotenv
from pathlib import Path

env_path=Path.cwd() / 'Chatbot' / '.env'#give the .env file path
print("Path to .env file:", env_path)
print("File exists:", env_path.exists())
# Load environment variables from .env file
load_dotenv(env_path,override=True)


#OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
api_key = os.getenv("OPENAI_API_KEY")
print("Loaded:", api_key is not None)

Note: you may need to restart the kernel to use updated packages.
Path to .env file: c:\Users\shara\OneDrive\Documents\Coding Stuff\Generative AI\Chatbot\.env
File exists: True
Loaded: True



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
#install langchain packages
%pip install langchain -qU
%pip install langchain-openai -qU

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#initialize the language model
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9, openai_api_key=api_key)

In [26]:
#Initialize the Prompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([("system", "You are an intelligent chatbot. Answer the following question."),#system message to set the context for the chatbot
("user", "{question}")])#user message with a variable {question} to take the user input

In [27]:
#initialize a output parswer to format the output in a specific way
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

In [28]:
#by using thes three components(llm, prompt, output_parser) we can create a chain to link them together and create a chatbot
chain=prompt | llm | output_parser

In [29]:
question = "What is the capital of France?"
response = chain.invoke({"question": question})
print(response)

The capital of France is Paris.


#initialize a prompt tempalte for dynamic interaction with the chatbot

In [39]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

#create a prompt template with a message placeholder for the question
prompt2 = ChatPromptTemplate.from_messages([SystemMessage(content="You are an intelligent chatbot. Answer the following question."),#system message to set the context for the chatbot
(MessagesPlaceholder(variable_name="question")), #message placeholder to take the user input, so can store many messages in the question variable and use it in the chatbot
])

In [47]:
chain2=prompt2 | llm | output_parser

In [48]:
question = "what is generative AI?"
response = chain2.invoke({"question": [HumanMessage(content=question)]})
print(response)

Generative AI refers to artificial intelligence systems that have the ability to create new data or content, such as images, text, or music, rather than just analyzing or processing existing data. These systems use algorithms to generate original and sometimes creative outputs based on patterns and information from the data they have been trained on. Generative AI can be used in a variety of applications, including generating realistic images, creating human-like text, and composing music.


Intialize a prompt Template with predefined conversation history

In [49]:
prompt3 = ChatPromptTemplate.from_messages([SystemMessage(content="You are an intelligent chatbot. Answer the following question."), 
                                            HumanMessage(content="My name is Sharadhi?"),
                                            AIMessage(content="Nice to meet you, How can I assist you today?"),
                                            MessagesPlaceholder(variable_name="question")
                                            ])

In [50]:
chain3=prompt3 | llm | output_parser

In [ ]:
# now it can answer to who am I , becuase it is given in HumanMessage in the prompt template, so it can understand the context and answer accordingly
question = "who am I?"
response = chain3.invoke({"question": [HumanMessage(content=question)]})
print(response)

You are Sharadhi.


Store the conversation and question history and then use it in prompt(without hardcoding like in previous example)

In [60]:
# to get the history of the conversation we can use the question variable which is a list of messages, so we can store the conversation in the question variable and use it in the chatbot
prompt4 = ChatPromptTemplate.from_messages([SystemMessage(content="You are an intelligent chatbot. Answer the following question."),
                                            MessagesPlaceholder(variable_name="history"),#message placeholder to store the conversation history in the conversation variable
                                            MessagesPlaceholder(variable_name="question")#message placeholder to take the user input, so can store many messages in the question variable and use it in the chatbot
                                            ])  

history = [HumanMessage(content="My name is Sharadhi"), AIMessage(content="Nice to meet you, How can I assist you today?")]# we can keep this history variable empty at the beginning if needed

In [62]:
chain4=prompt4 | llm | output_parser

In [63]:
question = "who am I?"
response = chain4.invoke({"history": history, "question": [HumanMessage(content=question)]})
print(response)

You are Sharadhi, it's a pleasure to interact with you.


In [64]:
#we have to update the history variable after every conversation to keep the conversation going, so we can append the user message and the chatbot response to the history variable after every conversation
history.extend([HumanMessage(content=question), AIMessage(content=response)])
history

[HumanMessage(content='My name is Sharadhi', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Nice to meet you, How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='who am I?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="You are Sharadhi, it's a pleasure to interact with you.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]